# Entropy-Gated Dynamic Budget Allocator

Capstone Architecture: How It WorksInstead of blindly letting an LLM generate tokens until it hits max_tokens, our controller monitors the step-wise Shannon Entropy $H(P_t)$ of the model's logits at every forward pass:

[ Input Prompt ]
                                 │
                                 ▼
                     ┌───────────────────────┐
                     │ Forward Pass (Step t) │
                     └───────────┬───────────┘
                                 │
                         Extract Logits
                                 │
                       Calculate Entropy H(P_t)
                                 │
            ┌────────────────────┴────────────────────┐
            ▼                                         ▼
   H(P_t) < Threshold                        H(P_t) >= Threshold
   (Model Has Converged)                    (Model Still Exploring)
            │                                         │
            ▼                                         ▼
[ HALT THINKING / TRANSITION ]               [ ALLOCATE NEXT TOKEN ]
   Save GPU Compute FLOPs                     (Keep In Thinking Loop)

Min Budget ($K_{\text{min}}$): Forces the model to think for at least $K_{\text{min}}$ steps to avoid premature answers on hard questions.Early Halt Condition ($H(P_t) < \tau_{\text{low}}$): If the entropy drops sharply, the model has reached its "Aha!" moment. We stop allocating thinking tokens immediately to save GPU FLOPs.Infinite Loop Halt ($t > K_{\text{max}}$): If the model plateaus in high entropy without converging, the controller forcefully cuts off the reasoning trajectory to prevent infinite generation.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class DynamicTestTimeComputeController:
    """
    Inference-time controller that monitors autoregressive token entropy
    and dynamically allocates or halts the thinking budget (K).
    """
    def __init__(
        self, 
        entropy_threshold_low: float = 1.2, 
        min_thinking_tokens: int = 5,
        max_thinking_budget: int = 30
    ):
        self.tau_low = entropy_threshold_low
        self.k_min = min_thinking_tokens
        self.k_max = max_thinking_budget

    def compute_step_entropy(self, logits: torch.Tensor) -> torch.Tensor:
        """
        Calculates Shannon Entropy H(P) over the vocabulary dimension.
        logits shape: [batch_size, vocab_size]
        returns shape: [batch_size]
        """
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1) # Numerically stable log
        entropy = -torch.sum(probs * log_probs, dim=-1)
        return entropy

    def generate_with_dynamic_budget(self, model: nn.Module, prompt_ids: torch.Tensor):
        """
        Runs autoregressive generation with real-time entropy monitoring.
        prompt_ids shape: [batch_size, seq_len]
        """
        batch_size = prompt_ids.shape[0]
        current_ids = prompt_ids.clone()
        
        # Track active status per batch item (True = Still Thinking, False = Halting)
        active_thinking = torch.ones(batch_size, dtype=torch.bool, device=prompt_ids.device)
        
        entropy_logs = [[] for _ in range(batch_size)]
        step = 0

        print(f"=== Starting Entropy-Gated Generation (Batch Size: {batch_size}) ===")

        while active_thinking.any() and step < self.k_max:
            step += 1
            
            # 1. Forward pass through the model
            with torch.no_grad():
                logits = model(current_ids) # [batch_size, current_seq_len, vocab_size]
                next_token_logits = logits[:, -1, :] # [batch_size, vocab_size]

            # 2. Measure current step entropy H(P_t)
            step_entropy = self.compute_step_entropy(next_token_logits)

            # Log step entropy for each sequence in the batch
            for b in range(batch_size):
                if active_thinking[b]:
                    entropy_logs[b].append(step_entropy[b].item())

            # 3. Check Halting Condition (if step >= k_min and H(P_t) < tau_low)
            if step >= self.k_min:
                converged_mask = step_entropy < self.tau_low
                # Deactivate sequences that have dropped below entropy threshold
                newly_halted = active_thinking & converged_mask
                active_thinking = active_thinking & (~converged_mask)
                
                for b in range(batch_size):
                    if newly_halted[b]:
                        print(f"  [Batch {b}] Step {step:02d}: Early Halt Triggered! Entropy dropped to {step_entropy[b].item():.3f} < {self.tau_low}")

            # 4. Sample next token
            probs = F.softmax(next_token_logits, dim=-1)
            next_tokens = torch.multinomial(probs, num_samples=1) # [batch_size, 1]

            # Append generated tokens
            current_ids = torch.cat([current_ids, next_tokens], dim=1)

            print(f"Step {step:02d} | Active Thinking Sequences: {active_thinking.sum().item()}/{batch_size} | Avg Batch Entropy: {step_entropy.mean().item():.3f}")

        print(f"\n=== Generation Complete. Total Budget Spent: {step} steps ===")
        return current_ids, entropy_logs


# =====================================================================
# Mock LLM Engine simulating variable entropy trajectories for testing
# =====================================================================
class MockLLM(nn.Module):
    def __init__(self, vocab_size=1000):
        super().__init__()
        self.vocab_size = vocab_size
        self.linear = nn.Linear(64, vocab_size)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        
        # Simulate hidden states
        hidden = torch.randn(batch_size, seq_len, 64)
        logits = self.linear(hidden)
        
        # Batch item 0: Fast convergence (entropy drops quickly after step 6)
        if seq_len > 10:
            logits[0, -1, 42] += (seq_len - 10) * 4.0
            
        # Batch item 1: Slow convergence / Hard question (entropy stays elevated)
        if seq_len > 18:
            logits[1, -1, 99] += (seq_len - 18) * 3.5
            
        return logits


# --- VERIFICATION EXECUTION ---
if __name__ == "__main__":
    torch.manual_seed(42)
    
    vocab_size = 1000
    mock_model = MockLLM(vocab_size=vocab_size)
    
    # Instantiate controller
    controller = DynamicTestTimeComputeController(
        entropy_threshold_low=1.5,
        min_thinking_tokens=5,
        max_thinking_budget=20
    )
    
    # Simulated prompt batch of 2 questions
    prompt_ids = torch.randint(0, vocab_size, (2, 4))
    
    output_ids, logs = controller.generate_with_dynamic_budget(mock_model, prompt_ids)

=== Starting Entropy-Gated Generation (Batch Size: 2) ===
Step 01 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.741
Step 02 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.712
Step 03 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.755
Step 04 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.749
Step 05 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.712
Step 06 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.721
Step 07 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.723
Step 08 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 6.679
Step 09 | Active Thinking Sequences: 2/2 | Avg Batch Entropy: 4.907
  [Batch 0] Step 10: Early Halt Triggered! Entropy dropped to 0.041 < 1.5
Step 10 | Active Thinking Sequences: 1/2 | Avg Batch Entropy: 3.370
Step 11 | Active Thinking Sequences: 1/2 | Avg Batch Entropy: 3.377
Step 12 | Active Thinking Sequences: 1/2 | Avg Batch Entropy: 3.336
Step 13 | Active Thinking Sequences: 1/2 | Avg Batch